In [8]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
import pandas as pd 
import numpy as np 
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.impute import SimpleImputer
torch.manual_seed(42) 
np.random.seed(42) 
print("Libraries imported successfully! PyTorch Version:", torch.__version__) 

Libraries imported successfully! PyTorch Version: 2.13.0+cpu


In [24]:
if not os.path.exists('house_prices.csv'):
     n = 1200
     demo_df = pd.DataFrame({
        'Sqft_Area': np.random.randint(500, 3500, n),
        'Bedrooms': np.random.randint(1, 6, n),
        'Bathrooms': np.random.randint(1, 4, n),
        'Age_Years': np.random.randint(0, 50, n),
        'Location_Tier': np.random.choice(['Urban', 'Suburban'], n),
        'Distance_to_City_km': np.random.uniform(0.5, 30.0, n),
        'Has_Garage': np.random.choice(['Yes', 'No'], n),
    })
     loc_num = (demo_df['Location_Tier'] == 'Urban').astype(int)
     garage_num = (demo_df['Has_Garage'] == 'Yes').astype(int)
     demo_df['Sale_Price'] = np.clip(
        50 + 0.12 * demo_df['Sqft_Area'] + 15 * demo_df['Bedrooms'] +
        10 * demo_df['Bathrooms'] - 1.5 * demo_df['Age_Years'] +
        20 * loc_num - 1.2 * demo_df['Distance_to_City_km'] +
        8 * garage_num + np.random.normal(0, 15, n), 40, 900
    )
     for col in ['Sqft_Area', 'Distance_to_City_km', 'Age_Years']:
        mask = np.random.rand(n) < 0.03
        demo_df.loc[mask, col] = np.nan
        demo_df.to_csv('house_prices.csv', index=False)
df = pd.read_csv('house_prices.csv')
df['Location_Tier'] = df['Location_Tier'].map({'Suburban': 0, 'Urban': 1})
df['Has_Garage'] = df['Has_Garage'].map({'No': 0, 'Yes': 1})
feature_cols = ['Sqft_Area','Bedrooms','Bathrooms','Age_Years','Location_Tier','Distance_to_City_km', 'Has_Garage']
X =df[feature_cols].values
y =df['Sale_Price'].values

X_encoded = X
y_encoded = y

imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

X_train,X_test,y_train,y_test = train_test_split(X_encoded,y_encoded,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

print(f"X_train shape: {X_train_tensor.shape},y_train shape: {y_train_tensor.shape}")


X_train shape: torch.Size([960, 7]),y_train shape: torch.Size([960, 1])


In [25]:
class HousePriceRegressor(nn.Module):
    def __init__(self):
       super(HousePriceRegressor,self).__init__()
       self.linear1 = nn.Linear(7, 32)
       self.linear2 = nn.Linear(32, 16)
       self.linear3 = nn.Linear(16, 1)
       self.ReLU = nn.ReLU()
    def forward(self,x):
        x = self.ReLU(self.linear1(x))
        x = self.ReLU(self.linear2(x))
        x = self.linear3(x)
        return x 
model = HousePriceRegressor()
print(model)
    

HousePriceRegressor(
  (linear1): Linear(in_features=7, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (linear3): Linear(in_features=16, out_features=1, bias=True)
  (ReLU): ReLU()
)


In [26]:
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("loss function: MSELoss | Optimizer: Adam")

loss function: MSELoss | Optimizer: Adam


In [27]:
num_epochs = 100 
batch_size = 32
print("Traning started...\n")

for epoch in range(num_epochs):
    for i in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[i:i+batch_size]
        y_batch = y_train_tensor[i:i+batch_size]
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
         print(f"Epoch [{epoch+1}/{num_epochs}],MSE Loss: {loss.item():.4f}")
         
print("\nTraining completed!")


Traning started...

Epoch [20/100],MSE Loss: nan
Epoch [40/100],MSE Loss: nan
Epoch [60/100],MSE Loss: nan
Epoch [80/100],MSE Loss: nan
Epoch [100/100],MSE Loss: nan

Training completed!


In [28]:
model.eval()
with torch.no_grad():
    test_predictions = model(X_test_tensor)
test_mse = loss_fn(test_predictions, y_test_tensor).item()
test_rmse = np.sqrt(test_mse)
print(f"Test Set Evaluation:")
print(f"  - Mean Squared Error (MSE)      : {test_mse:.4f}")
print(f"  - Root Mean Squared Error (RMSE): {test_rmse:.4f}\n")
print(f"{'Actual':<15} {'Predicted':<15} {'Abs Error':<15}")
print("-" * 45)
with torch.no_grad():
    for i in range(5):
        actual = y_test_tensor[i].item()
        predicted = test_predictions[i].item()
        error = abs(actual - predicted)
        print(f"{actual:<15.2f} {predicted:<15.2f} {error:<15.2f}")


Test Set Evaluation:
  - Mean Squared Error (MSE)      : nan
  - Root Mean Squared Error (RMSE): nan

Actual          Predicted       Abs Error      
---------------------------------------------
161.52          nan             nan            
168.79          nan             nan            
401.31          nan             nan            
152.91          nan             nan            
294.35          nan             nan            


In [30]:
raw_house = np.array([[1800, 3, 2, 8, 1, 4.5, 1]])
imputed_data = imputer.transform(raw_house)
scaled_data = scaler.transform(imputed_data)
house_tensor = torch.tensor(scaled_data,dtype=torch.float32)
model.eval()
with torch.no_grad():
    predicted_price = model(house_tensor).item()
print(f"Predicted Sale Price: ${predicted_price:.2f}k")
    

Predicted Sale Price: $nank
